# GridVision Electricity Anomaly Detection and Spatial Localization

This notebook implements the methodology in `docs/GridVision_Electricity_Anomaly_Detection_and_Spatial_Localization.docx`: temporal baseline → point/sequence detection using time-domain, FFT, and Haar-wavelet features → anomalous-window characterization with time-domain, FFT, and wavelet features → rule-based type assignment. The binary detector learns normal-vs-anomaly from normal training rows and controlled synthetic training events; there is no machine-learning anomaly-type classifier. Evaluation uses held-out customers and future windows, followed by geographic grouping.

**Leakage controls:** source label/metadata columns are never read as model inputs; customer IDs are partitioned before sampling; time is split chronologically; point features use only current and strictly prior readings; the detector is fit with training customers/time only, its alert threshold and rule cutoffs are selected from validation synthetic events and training-normal readings, and test data is never used for fitting or threshold selection. Synthetic evaluation labels remain separate from features. Because the source data is normal-only, synthetic test events are a controlled proxy—not real-world anomaly ground truth.

## 1. Configure the experiment and imports

The sample limits bound memory while retaining disjoint, complete customer groups. Increase them only after checking available RAM and runtime.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from gridvision_pipeline import GridVisionExperiment, PipelineConfig

config = PipelineConfig(
    parquet_path=str(ROOT / 'data' / 'synthetic_2019_normal.parquet'),
    model_path=str(ROOT / 'models' / 'gridvision_anomaly_model.joblib'),
    report_dir=str(ROOT / 'reports'),
    random_seed=42,
    train_customer_sample=200,
    validation_customer_sample=100,
    test_customer_sample=100,
    training_score_sample=200_000,
    training_events_per_class=150,
    evaluation_events_per_class=50,
    false_positive_quantile=0.995,
)
experiment = GridVisionExperiment(config)
print('Input:', config.parquet_path)
print('Output model:', config.model_path)

## 2. Audit the parquet schema and create leakage-safe splits

The file has 87.6 million hourly rows. The loader reads only customer ID, timestamp, energy, and coordinates. Any supplied `is_anomaly`, `anomaly_type`, `original_energy_kwh`, event metadata, and severity fields are intentionally excluded. First, all customer IDs are divided into 80/10/10 train/validation/test partitions; only then are bounded customer samples selected. Each split also uses a different chronological window: first 70%, next 15%, final 15%.

In [ ]:
audit = experiment.audit_and_load()
print(json.dumps(audit, indent=2, default=str))
assert experiment.split_integrity['customer_overlap'] == 0

## 3. Engineer causal time-domain, FFT, and wavelet features; train the binary detector

The robust baseline is formed from trailing 24/168-hour history with a one-hour shift. Each hourly point is characterized using the trailing 25-hour residual window: FFT dominant frequency, spectral energy/entropy/centroid, peak magnitude and low/high-band energy; plus multi-scale Haar detail energy, variance, magnitude, entropy, and dominant scale. These signal features join residual/deviation, lag, variability, zero/flatline-run and calendar features as direct inputs to a **binary normal-vs-anomaly detector**. Its supervised training examples are normal training points plus controlled injected training anomalies. Spike/drop decision cutoffs are learned from normal training residuals. The detector threshold is selected using validation events; clean validation data is used to report false-alert rate. Test data is not used for fitting or selection.

In [ ]:
detector_summary = experiment.fit_binary_detector()
print(json.dumps(detector_summary, indent=2, default=str))

## 4. Calibrate anomaly detection and deterministic type rules

The source contains no positive anomaly examples. Therefore, spike, drop, sustained-zero, and flatline events are injected into a copy of training customers only to train the binary detector. All anomaly types share the same positive detector target; type labels are not model outputs. Four deterministic rules characterize a detected signal: sustained zeros → zero block; repeated constant values despite prior variation → flatline; positive residual past the normal-trained upper cutoff → spike; negative residual past the lower cutoff → drop. Validation events are generated independently on validation customers to choose the detector threshold and assess the rules.

In [ ]:
calibration_summary = experiment.calibrate_detector_and_rules()
print(json.dumps(calibration_summary, indent=2, default=str))

## 5. Evaluate on held-out customers and the final chronological window

Test events are independently injected into held-out test customers. Detection is measured at the point level with precision, recall, F1, PR-AUC, and false alerts per customer. The deterministic type rules are reported with macro-F1, per-class results, and a confusion matrix. These scores quantify performance on the stated synthetic event generator, not on naturally occurring grid faults.

In [ ]:
test_summary = experiment.evaluate_held_out_test()
print(json.dumps(test_summary, indent=2, default=str))

### Prediction input columns and detector feature contract

At inference, provide five raw columns: `OBJECTID`, `timestamp`, `normalized_building_energy_kwh`, `latitude`, and `longitude`. Of these, only the engineered 30-feature vector below is passed into the binary detector. Customer ID is for grouping, and coordinates are retained for geographic localization; neither is a detector predictor. The event-window feature vector is computed after detection for reporting and deterministic type rules, not passed to a multiclass model. Source labels and `original_energy_kwh` are excluded.

In [ ]:
from gridvision_pipeline import INPUT_COLUMNS, POINT_FEATURES, EVENT_FEATURES

print('Raw prediction input columns (required):')
print('\n'.join(f'- {column}' for column in INPUT_COLUMNS))
print(f'\nBinary detector predictors ({len(POINT_FEATURES)} engineered features):')
print('\n'.join(f'- {feature}' for feature in POINT_FEATURES))
print(f'\nPost-detection event characterization features ({len(EVENT_FEATURES)}; not ML predictors):')
print('\n'.join(f'- {feature}' for feature in EVENT_FEATURES))

## 6. Prepare anomaly windows and localize spatial-temporal events

The existing detector and deterministic anomaly-type rules are unchanged. The event stage consumes their per-customer detected windows (`customer_id`, start/end, rule-assigned anomaly type, coordinates, and maximum score), then adds H3 cells and DBSCAN geographic clusters. DBSCAN runs on unique customer locations so repeat alerts from one customer cannot satisfy the multi-customer density requirement. Temporal correlation uses configurable start-time difference, overlap, and interval-gap limits. Dense geographic clusters are combined into a `MULTI_AREA` event only when their intervals are also temporally related; spatial noise remains customer-level rather than being grouped just because it happened at the same time. Ground-truth injected-event IDs/labels are excluded from grouping and are used only in the following evaluation cell. Geographic clusters do not identify feeders, transformers, or substations.

In [ ]:
localized_events = experiment.localize()
print('Per-customer anomaly windows:', len(experiment.localized_records))
print('Spatial-temporal events:', len(localized_events))
print('Localization parameters:')
print(json.dumps({
    'h3_resolution': config.h3_resolution,
    'spatial_radius_km': config.spatial_radius_km,
    'dbscan_min_customers': config.spatial_dbscan_min_customers,
    'maximum_start_time_difference_minutes': config.maximum_start_time_difference_minutes,
    'minimum_temporal_overlap': config.minimum_temporal_overlap,
    'maximum_interval_gap_minutes': config.maximum_interval_gap_minutes,
}, indent=2))
from IPython.display import display
print('Customer-level localized records:')
display(experiment.localized_records.head())
print('Event-level summary:')
display(localized_events.head())

### 6.1 Validate reconstructed events against held-out injected events

Event matching occurs only after localization. Precision, recall, F1, mean detection delay, event purity, false events, and missed events compare reconstructed events with held-out synthetic ground truth. The synthetic generator injects customer-level events independently; it does not provide authoritative common outage/local-area or multi-area event labels, so these metrics cannot validate real common-event scope.

In [ ]:
print(json.dumps(experiment.event_localization_metrics, indent=2, default=str))
localized_events[[
    'event_id', 'event_scope', 'event_start', 'event_end',
    'affected_customer_count', 'dominant_anomaly_type',
    'spatial_cluster_count', 'latitude_center', 'longitude_center',
    'spatial_extent_km', 'mean_anomaly_score', 'max_anomaly_score',
]].head()

### 6.2 Inspect geographic, timeline, event-size, and scope plots

The localization stage saves a customer map, event map, selected-event timeline, event-size distribution, and event-scope distribution under `reports/event_visualizations/`.

In [ ]:
from IPython.display import Image, display

for plot_name, plot_path in experiment.event_visualizations.items():
    print(plot_name, '->', plot_path)
    display(Image(filename=plot_path))

## 7. Save the trained model and evaluation artifacts

The joblib bundle contains the unchanged binary detector and rule thresholds. Reports separately save predicted anomaly windows, customer-level event membership, event summaries, event-localization metrics, the configurable spatial-temporal settings, and all five validation plots. Ground-truth synthetic labels are used only by the evaluation metrics after event grouping.

In [ ]:
artifacts = experiment.save_artifacts()
print(json.dumps(artifacts, indent=2))
assert Path(artifacts['model']).is_file()
assert Path(artifacts['metrics']).is_file()
assert Path(artifacts['spatial_event_records']).is_file()
assert Path(artifacts['spatial_temporal_events']).is_file()
assert Path(artifacts['event_localization_evaluation']).is_file()

## 8. Interpretation and limitations

- The original file is normal-only; anomaly precision/recall and classification scores are synthetic-injection evaluations.
- Customer and time holdouts reduce identity and temporal leakage, but the default results use bounded samples (200 train, 100 validation, 100 test customers), not all 10,000 customers.
- The type output is rule-based, not a learned multiclass model. Synthetic-template classification results are not real-fault performance.
- The inputs use normalized building energy. The model's values and alert thresholds should be recalibrated before use with raw kWh or a different population.
- Geographic DBSCAN clusters are not feeder/transformer localization. Real operational validation requires labeled events and network topology.

See `docs/GRIDVISION_MODEL.md` for setup, execution, artifact schema, and deployment guidance. The raw prediction inputs, all 30 detector feature names, all 51 post-detection event features, and explicitly excluded labels are also listed in `docs/MODEL_INPUT_COLUMNS.txt`.